In [1]:
import sys
import pandas as pd
from pathlib import Path

In [2]:
current_dir = Path.cwd()
utils_path = next(
    (
        p
        for p in [current_dir] + list(current_dir.parents)
        if (p / "notebook_utils.py").exists()
    ),
    None,
)

if utils_path is None:
    raise FileNotFoundError("notebook_utils.py не найден!")

sys.path.append(str(utils_path))

In [3]:
from notebook_utils import (
    setup_env,
    load_data_for_modeling,
    get_exp_manager
)

PROJECT_ROOT, config = setup_env()
exp_manager = get_exp_manager()

print(f"Project Root: {PROJECT_ROOT}")

Project Root: D:\Education\Arcticle\dtp_project


## Модель

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from notebook_utils import setup_env, load_data_for_modeling, get_exp_manager, save_catboost_model, load_catboost_model
from etl.enricher import EconomicEnricher

In [11]:
exp_manager = get_exp_manager()
PROJECT_ROOT, config = setup_env()

df_dtp = load_data_for_modeling(config, PROJECT_ROOT, data_source="full_df", need_drop=False)
df_econ = pd.read_excel(PROJECT_ROOT / config['paths']['econom_data'])

enricher = EconomicEnricher(config)
df_final = enricher.merge(df_dtp, df_econ)

Загрузка данных из: D:\Education\Arcticle\dtp_project\data\processed\dtp_full_dataset.parquet
Режим таргета: binary_severe. Распределение:
target
0    0.566
1    0.434
Name: proportion, dtype: float64


2026-02-09 12:14:27,105 - INFO - === НАЧАЛО ОБОГАЩЕНИЯ ДАННЫХ ===
2026-02-09 12:14:27,107 - INFO - --- [Enricher] 1. Очистка экономических данных ---
2026-02-09 12:14:27,112 - INFO -    Колонка 'RTA_dead': заполнено 104 пропусков (median=19.0)
2026-02-09 12:14:27,116 - INFO -    Колонка 'RTA_serious': заполнено 103 пропусков (median=105.0)
2026-02-09 12:14:27,120 - INFO -    Колонка 'RTA_minor': заполнено 103 пропусков (median=298.0)
2026-02-09 12:14:27,128 - INFO -    Уникальных регионов после очистки: 68
2026-02-09 12:14:27,129 - INFO - --- [Enricher] 2. Агрегация дубликатов ---
2026-02-09 12:14:27,131 - INFO -    Размер ДО агрегации: (1000, 14)
2026-02-09 12:14:27,135 - INFO -    Размер ПОСЛЕ агрегации: (680, 11)
2026-02-09 12:14:27,137 - INFO - --- [Enricher] 3. Сопоставление регионов (Matching) ---
2026-02-09 12:14:28,047 - INFO -    Сматчилось уникальных регионов: 2092/2237 (93.5%)
2026-02-09 12:14:28,123 - WARNING -    Топ-5 ненайденных регионов: ['чукотский_автономный_округ_чук

In [12]:
target_col = config['features']['target_col']
cat_cols = config['features'].get('cat_cols_with_econom', ["region_id", "region_parent", "light_cat", "scheme", "category"])

cat_cols = [c for c in cat_cols if c in df_final.columns]
print(f"Категориальные признаки: {cat_cols}")

drop_explicit = [target_col, 'year', 'date', 'datetime', 'join_year']

object_cols = df_final.select_dtypes(include=['object']).columns.tolist()
text_garbage = [c for c in object_cols if c not in cat_cols]

if text_garbage:
    print(f"Будут удалены технические текстовые колонки: {text_garbage}")

final_drop = [c for c in drop_explicit + text_garbage if c in df_final.columns]

X = df_final.drop(columns=final_drop)
y = df_final[target_col]

for col in cat_cols:
    X[col] = X[col].astype(str).fillna("Missing")

Категориальные признаки: ['region_id', 'region_parent', 'light_cat', 'scheme', 'category']


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (1172705, 97), Test: (293177, 97)


In [14]:
MODEL_NAME = "01_catboost"
STAGE_NAME = "full_with_region_2000"
FORCE_RETRAIN = False

model = None

if not FORCE_RETRAIN:
    try:
        model = load_catboost_model(exp_manager, MODEL_NAME, stage=STAGE_NAME)
        print("✅ Модель загружена с диска.")
    except:
        print("Модель не найдена, будем обучать.")

if model is None:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights_dict = dict(zip(classes, weights))
    
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    test_pool = Pool(X_test, y_test, cat_features=cat_cols)
    
    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.08, # Чуть меньше LR, т.к. фичей стало больше
        depth=6,
        loss_function='Logloss',
        eval_metric='AUC',
        class_weights=class_weights_dict,
        task_type="CPU", # Или GPU
        early_stopping_rounds=100,
        verbose=100
    )
    
    model.fit(train_pool, eval_set=test_pool, use_best_model=True)
    save_catboost_model(model, exp_manager, MODEL_NAME, stage=STAGE_NAME)

Модель не найдена, будем обучать.
0:	test: 0.7224034	best: 0.7224034 (0)	total: 944ms	remaining: 31m 27s
100:	test: 0.7680689	best: 0.7680689 (100)	total: 1m 20s	remaining: 25m 19s
200:	test: 0.7734272	best: 0.7734272 (200)	total: 2m 41s	remaining: 24m 6s
300:	test: 0.7764064	best: 0.7764064 (300)	total: 4m 2s	remaining: 22m 48s
400:	test: 0.7782219	best: 0.7782219 (400)	total: 5m 26s	remaining: 21m 42s
500:	test: 0.7793792	best: 0.7793792 (500)	total: 6m 48s	remaining: 20m 22s
600:	test: 0.7802220	best: 0.7802220 (600)	total: 8m 12s	remaining: 19m 6s
700:	test: 0.7810283	best: 0.7810286 (699)	total: 9m 34s	remaining: 17m 44s
800:	test: 0.7817793	best: 0.7817793 (800)	total: 10m 57s	remaining: 16m 23s
900:	test: 0.7822451	best: 0.7822451 (900)	total: 12m 19s	remaining: 15m 2s
1000:	test: 0.7827925	best: 0.7827925 (1000)	total: 13m 42s	remaining: 13m 40s
1100:	test: 0.7831509	best: 0.7831509 (1100)	total: 15m 8s	remaining: 12m 22s
1200:	test: 0.7834493	best: 0.7834494 (1199)	total: 16m 

In [15]:
# --- 7. Результаты ---
y_proba = model.predict_proba(X_test)[:, 1]
print(f"\nROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

# Важность признаков
feature_importance = model.get_feature_importance(prettified=True)
print("\nТоп-10 важных признаков:")
print(feature_importance.head(10))

# Посмотрим, попала ли экономика в топ
econ_cols = ['Avg_Salary', 'Population', 'GAP', 'Street_Length', 'region_parent']
print("\nПозиции экономических факторов:")
print(feature_importance[feature_importance['Feature Id'].isin(econ_cols)])


ROC AUC: 0.7851

Топ-10 важных признаков:
       Feature Id  Importances
0       region_id    21.530793
1          scheme     9.153055
2    viol_runaway     3.995750
3       ppl_count     3.973277
4   ppl_ped_count     3.795587
5    vh_count_car     3.782493
6     RTA_serious     3.282413
7        category     2.872445
8  ppl_pass_count     2.780055
9      viol_drunk     2.738902

Позиции экономических факторов:
       Feature Id  Importances
11     Avg_Salary     2.183256
16  region_parent     1.724794
20            GAP     0.976924
29     Population     0.709671
32  Street_Length     0.670191


In [16]:
STAGE_NAME = "00_ab_test_econom"  # Новая папка для этого сравнения
FORCE_RETRAIN = True             # Поставь True, чтобы переобучить обе модели заново

# --- 2. Функция подготовки данных (Preprocessing) ---
def prepare_dataset(df, cat_cols, target_col):
    """Чистит датасет: удаляет мусор, оставляет фичи, готовит X и y."""
    df = df.copy()
    
    # 1. Бинаризация таргета
    target_map = {0: 0, 1: 1, 2: 1}
    df['target_bin'] = df[target_col].map(target_map)
    
    # 2. Определяем, что удалять
    # Явно ненужные колонки
    drop_explicit = [target_col, 'year', 'date', 'datetime', 'join_year', 'target_bin']
    
    # Текстовый мусор (все object колонки, которые НЕ в списке категорий)
    object_cols = df.select_dtypes(include=['object']).columns.tolist()
    text_garbage = [c for c in object_cols if c not in cat_cols]
    
    # Финальный список на удаление
    final_drop = [c for c in drop_explicit + text_garbage if c in df.columns]
    
    X = df.drop(columns=final_drop)
    y = df['target_bin']
    
    # 3. Заполняем пропуски в категориях
    for col in cat_cols:
        if col in X.columns:
            X[col] = X[col].astype(str).fillna("Missing")
            
    return X, y

# --- 3. Загрузка данных ---
print("--- [1] Загрузка данных ---")
# Грузим исходник один раз
df_raw = load_data_for_modeling(config, PROJECT_ROOT, data_source="full_df", need_drop=False)
df_econ_raw = pd.read_excel(PROJECT_ROOT / config['paths']['econom_data'])

# --- 4. Формирование двух датасетов ---
print("\n--- [2] Подготовка датасетов ---")

# КОНФИГУРАЦИЯ 1: BASELINE (Без экономики)
cat_cols_base = ["region_id", "light_cat", "scheme", "category"]
# Используем копию raw, чтобы не испортить
X_base, y_base = prepare_dataset(df_raw, cat_cols_base, config['features']['target_col'])
print(f"Dataset Baseline: {X_base.shape}")

# КОНФИГУРАЦИЯ 2: ENRICHED (С экономикой)
print("   Запуск Enricher...")
enricher = EconomicEnricher(config)
df_enriched_raw = enricher.merge(df_raw.copy(), df_econ_raw) # Важно передать копию!

cat_cols_econ = ["region_id", "region_parent", "light_cat", "scheme", "category"]
X_econ, y_econ = prepare_dataset(df_enriched_raw, cat_cols_econ, config['features']['target_col'])
print(f"Dataset Enriched: {X_econ.shape}")

# --- 5. Функция обучения/загрузки ---
def train_or_load_model(model_name, X, y, cat_features):
    print(f"\n🚀 Работаем с моделью: {model_name}")
    
    # Split (фиксированный random_state для честности сравнения)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    model = None
    
    # 1. Попытка загрузки
    if not FORCE_RETRAIN:
        try:
            model = load_catboost_model(exp_manager, model_name, stage=STAGE_NAME)
            print("   ✅ Загружена с диска.")
        except:
            print("   ⚠️ Не найдена, будем обучать.")
            
    # 2. Обучение
    if model is None:
        print("   ⏳ Обучение...")
        # Баланс классов
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
        class_weights_dict = dict(zip(classes, weights))
        
        train_pool = Pool(X_train, y_train, cat_features=cat_features)
        test_pool = Pool(X_test, y_test, cat_features=cat_features)
        
        model = CatBoostClassifier(
            iterations=1500, # 1500 достаточно для сравнения
            learning_rate=0.1,
            depth=6,
            loss_function='Logloss',
            eval_metric='AUC',
            class_weights=class_weights_dict,
            task_type="CPU",
            early_stopping_rounds=50,
            verbose=200,
            allow_writing_files=False
        )
        
        model.fit(train_pool, eval_set=test_pool, use_best_model=True)
        save_catboost_model(model, exp_manager, model_name, stage=STAGE_NAME)
        
    # 3. Оценка
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    print(f"   📊 ROC AUC: {auc:.5f}")
    
    return model, auc

# --- 6. ЗАПУСК СРАВНЕНИЯ ---
model_base, auc_base = train_or_load_model(
    "baseline_no_econ", X_base, y_base, cat_cols_base
)

model_econ, auc_econ = train_or_load_model(
    "enriched_with_econ", X_econ, y_econ, cat_cols_econ
)

# --- 7. ИТОГОВЫЙ ОТЧЕТ ---
print("\n" + "="*40)
print("🏁 РЕЗУЛЬТАТЫ A/B ТЕСТИРОВАНИЯ")
print("="*40)
print(f"1. Baseline (только DTP): {auc_base:.5f}")
print(f"2. Enriched (DTP + Econ): {auc_econ:.5f}")

diff = auc_econ - auc_base
if diff > 0:
    print(f"\n✅ УЛУЧШЕНИЕ: +{diff:.5f} AUC")
    print("Вывод: Экономические данные добавляют полезный сигнал!")
else:
    print(f"\n❌ УХУДШЕНИЕ/ШУМ: {diff:.5f} AUC")
    print("Вывод: Экономика не помогает или вносит шум.")

# --- 8. Анализ важности (только для Enriched) ---
print("\n--- Топ-5 признаков (Enriched) ---")
fi = model_econ.get_feature_importance(prettified=True)
print(fi.head(10))

econ_cols_check = ['Avg_Salary', 'Population', 'GAP', 'region_parent']
print("\n--- Позиции экономических факторов ---")
print(fi[fi['Feature Id'].isin(econ_cols_check)])

--- [1] Загрузка данных ---
Загрузка данных из: D:\Education\Arcticle\dtp_project\data\processed\dtp_full_dataset.parquet
Режим таргета: binary_severe. Распределение:
target
0    0.566
1    0.434
Name: proportion, dtype: float64

--- [2] Подготовка датасетов ---
Dataset Baseline: (1465882, 87)
   Запуск Enricher...


2026-02-09 13:12:14,273 - INFO - === НАЧАЛО ОБОГАЩЕНИЯ ДАННЫХ ===
2026-02-09 13:12:14,276 - INFO - --- [Enricher] 1. Очистка экономических данных ---
2026-02-09 13:12:14,295 - INFO -    Колонка 'RTA_dead': заполнено 104 пропусков (median=19.0)
2026-02-09 13:12:14,300 - INFO -    Колонка 'RTA_serious': заполнено 103 пропусков (median=105.0)
2026-02-09 13:12:14,306 - INFO -    Колонка 'RTA_minor': заполнено 103 пропусков (median=298.0)
2026-02-09 13:12:14,320 - INFO -    Уникальных регионов после очистки: 68
2026-02-09 13:12:14,321 - INFO - --- [Enricher] 2. Агрегация дубликатов ---
2026-02-09 13:12:14,322 - INFO -    Размер ДО агрегации: (1000, 14)
2026-02-09 13:12:14,343 - INFO -    Размер ПОСЛЕ агрегации: (680, 11)
2026-02-09 13:12:14,345 - INFO - --- [Enricher] 3. Сопоставление регионов (Matching) ---
2026-02-09 13:12:15,613 - INFO -    Сматчилось уникальных регионов: 2092/2237 (93.5%)
2026-02-09 13:12:15,802 - WARNING -    Топ-5 ненайденных регионов: ['чукотский_автономный_округ_чук

Dataset Enriched: (1465882, 97)

🚀 Работаем с моделью: baseline_no_econ
   ⏳ Обучение...
0:	test: 0.7234918	best: 0.7234918 (0)	total: 677ms	remaining: 16m 54s
200:	test: 0.7724686	best: 0.7724686 (200)	total: 2m 55s	remaining: 18m 56s
400:	test: 0.7755615	best: 0.7755615 (400)	total: 6m 16s	remaining: 17m 11s
600:	test: 0.7768818	best: 0.7768818 (600)	total: 9m 9s	remaining: 13m 42s
800:	test: 0.7776083	best: 0.7776083 (800)	total: 12m 7s	remaining: 10m 35s
1000:	test: 0.7780904	best: 0.7780904 (1000)	total: 15m 7s	remaining: 7m 32s
1200:	test: 0.7784497	best: 0.7784497 (1200)	total: 17m 58s	remaining: 4m 28s
1400:	test: 0.7787251	best: 0.7787263 (1395)	total: 20m 51s	remaining: 1m 28s
1499:	test: 0.7788651	best: 0.7788651 (1499)	total: 22m 16s	remaining: 0us

bestTest = 0.7788650598
bestIteration = 1499

CatBoost сохранен: D:\Education\Arcticle\dtp_project\res\02_econom_data\models\00_ab_test_econom\baseline_no_econ\model.cbm
   📊 ROC AUC: 0.77887

🚀 Работаем с моделью: enriched_with